本项目使用 **UD 2.15 English-EWT** 数据集，实现基于图的依存句法分析器。
包括词嵌入在内的全部神经网络参数均随机初始化，不使用预训练模型或现成的依存分析器。
对比三组模型：MLP 基线、仅使用词形的 BiLSTM 模型，以及同时使用词形和词性的 BiLSTM 模型。
自行实现 Chu–Liu/Edmonds 解码，并显式保证每句只有一个根词且不存在环。

## 环境准备

In [ ]:
from pathlib import Path
import os, sys, json, inspect
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "请从项目根目录或 notebooks 目录启动。"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown, Image, Code
from src.data import read_conllu, build_vocabs, ParserDataset, collate, dataset_statistics
from src.model import Parser, ModelConfig, Biaffine
from src.engine import seed_everything, infer, attachment_scores, train, Predictor
from src.decoder import decode_mst, validate_tree
seed_everything(42)
torch.set_num_threads(2)
pd.set_option("display.max_columns", 12)

中文列名 = {
    "split": "数据划分", "sentences": "句子数", "words": "词数", "mean_length": "平均句长",
    "max_length": "最大句长", "oov_percent": "未登录词比例（%）", "ID": "编号", "FORM": "词形",
    "LEMMA": "词元", "UPOS": "通用词性", "XPOS": "细粒度词性", "FEATS": "形态特征",
    "HEAD": "父节点", "DEPREL": "依存关系", "DEPS": "增强依存", "MISC": "其他信息",
    "Relation": "依存关系", "Training tokens": "训练集词数", "Model": "模型", "Official LAS": "官方 LAS",
    "Parameters": "参数量", "Training seconds": "训练耗时（秒）", "Tokens/second": "每秒处理词数",
    "Best epoch": "最佳轮次", "Dev LAS": "验证集 LAS", "tokens": "词数", "Tokens": "词数",
    "word_dim": "词嵌入维度", "pos_dim": "词性嵌入维度", "hidden": "单向隐藏层维度",
    "layers": "编码层数", "arc_dim": "弧评分维度", "rel_dim": "关系评分维度", "dropout": "丢弃率",
    "name": "模型名称", "parameters": "参数量", "training_seconds": "训练耗时（秒）",
    "best_epoch": "最佳轮次", "best_dev_LAS": "最佳验证 LAS", "epochs_run": "实际轮数", "device": "设备",
    "python": "Python 版本", "torch": "PyTorch 版本", "cuda": "CUDA 版本", "gpu": "显卡",
    "parallel_training": "是否并行训练", "distance_group": "依存距离分组", "length_group": "句长分组",
    "relation": "依存关系", "oov": "是否未登录词", "position": "位置", "word": "词形", "pos": "词性",
    "rel": "依存关系", "upos": "通用词性", "gold_head": "真实父节点", "pred_head": "预测父节点",
    "gold_rel": "真实关系", "pred_rel": "预测关系", "head_error": "父节点错误", "relation_error": "关系错误",
    "lengths": "序列长度", "heads": "父节点", "rels": "关系", "mask": "有效位置掩码", "dep_mask": "依存词掩码"
}
中文取值 = {"train": "训练集", "dev": "验证集", "test": "测试集", "baseline": "MLP 基线",
    "notebook_demo": "单轮教学模型", "word_only": "仅词形模型", "word_pos": "词形与词性模型", "all_full": "全部词、完整标签",
    "all_coarse": "全部词、去除关系子类型", "no_punct_full": "排除标点、完整标签",
    "Short (1-10)": "短句（1–10 词）", "Medium (11-25)": "中句（11–25 词）", "Long (26+)": "长句（26 词及以上）"}
def 中文展示(value):
    # 仅翻译界面标签，保留原始英文词形与标准依存标签。
    if isinstance(value, pd.DataFrame):
        value = value.copy()
        for column in ("split", "Model", "指标口径", "name", "length_group"):
            if column in value:
                value[column] = value[column].replace(中文取值)
        value = value.rename(columns=中文列名, index=中文列名)
    elif isinstance(value, pd.Series):
        value = value.rename(index=中文列名)
    display(value)

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
print("运行设备：", "CUDA" if torch.cuda.is_available() else "CPU")
RESULTS = ROOT / "experiments/full"
RUN_FULL_TRAINING = False

##  数据集读取
仅使用 `r2.15` 版本，固定到提交 `4dc8e10cf32352e11ab2c46e024b19853b91546e`。
清单记录数据与官方评估脚本的 SHA-256 摘要，以检查文件是否一致。
词表只根据训练集建立；验证集 LAS 用于选择断点；模型选择固定后才使用测试标签进行最终评估。

参考来源：[English-EWT r2.15 数据](https://github.com/UniversalDependencies/UD_English-EWT/tree/r2.15)、
[CoNLL-U 格式说明](https://universaldependencies.org/format.html)。

In [ ]:
from scripts.download_data import download
from src.experiment import verify_data
if not (ROOT / "data/manifest.json").exists():
    download()
manifest = verify_data()
splits = {name: read_conllu(ROOT / f"data/raw/en_ewt-ud-{name}.conllu")
          for name in ("train", "dev", "test")}
vocabs = build_vocabs(splits["train"])
print("数据版本：", manifest["release"], "| 树库：", manifest["treebank"])
print("通过校验的数据及评估文件数：", len(manifest["files"]))
中文展示(pd.DataFrame(dataset_statistics(splits, vocabs)).round(3))

整数编号的行表示真实句法词。多词条目保留为表层元信息，小数编号的空节点不参与基本依存树训练。
文件中的词编号会显式映射为模型内部连续位置，人工根节点 ROOT 的位置为 0，且没有父节点。
以下展示训练集中的第一个句子。英文语料原文及 UD 标准标签保持原样。

In [ ]:
sentence = splits["train"][0]
print(sentence.text)
中文展示(pd.DataFrame(sentence.tokens, columns=["ID", "FORM", "LEMMA", "UPOS", "XPOS",
                                              "FEATS", "HEAD", "DEPREL", "DEPS", "MISC"]))

## 数据探索
统计句子长度、训练集词性分布及依存关系频率。
未登录词比例以小写词形和仅由训练集建立的词表为依据。
此处对数据作描述性统计，不依据测试集选择特征或模型。

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Micro Hei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for name, sentences in splits.items():
    axes[0].hist([len(s) for s in sentences], bins=range(0, 101, 5), alpha=.45, label=中文取值.get(name, name))
axes[0].set(title="句子长度分布（显示至 100 词）", xlabel="句法词数", ylabel="句子数量")
axes[0].legend()
for ax, column, title in [(axes[1], 3, "训练集通用词性分布"), (axes[2], 7, "训练集高频依存关系（前 12 类）")]:
    counts = Counter(row[column] for s in splits["train"] for row in s.tokens)
    pd.Series(dict(counts.most_common(12))).plot.bar(ax=ax, rot=50)
    ax.set(title=title, ylabel="词数")
fig.tight_layout()
中文展示(fig)
plt.close(fig)
print("词表大小：", {中文列名.get(key, key): len(value) for key, value in vocabs.items()})
中文展示(pd.DataFrame(Counter(r[7] for s in splits["train"] for r in s.tokens).most_common(),
                     columns=["Relation", "Training tokens"]))

## 5. 特征处理
查词表前将词形转换为小写。词形和词性词表中的 PAD、UNK、ROOT 编号分别为 0、1、2。
未见词使用通过训练学习到的 UNK 向量，训练中的随机词替换为其提供学习信号。
依存关系保留训练集中出现的全部子类型。

掩码将填充位置排除在候选父节点之外，并将 ROOT 和填充位置排除在损失计算之外。
**使用词性的模型读取金标准 UPOS。** 本实验采用金标准分词，评估范围是依存分析。

In [ ]:
dataset = ParserDataset(splits["train"][:3], vocabs)
batch = collate([dataset[0], dataset[1], dataset[2]])
中文展示(pd.DataFrame({key: [tuple(value.shape)] for key, value in batch.items()
                      if isinstance(value, torch.Tensor)}))
print("第一句的词形编号：", batch["words"][0, :12].tolist())
print("第一句的父节点位置：", batch["heads"][0, :12].tolist())
print("参与损失计算的词数：", int(batch["dep_mask"].sum()))

## 模型结构
对第 $i$ 个词，输入表示为 $x_i=[e_w(w_i);e_p(p_i)]$；仅词形模型省略词性嵌入 $e_p$。
两层双向 LSTM 同时编码人工根节点和真实词：
$$h_i=operatorname{BiLSTM}(x_0,ldots,x_n)_i.$$
不同的 MLP 分别产生依存词与父节点表示，依存弧评分为：
$$s(i,j)=[d_i;1]^	op U[g_j;1]
  =d_i^	op U_0g_j+a^	op d_i+b^	op g_j+c.$$
依存关系使用独立的 MLP，并为每个关系类别配置一个双仿射矩阵：
$$s_{rel}(i,j,r)=[d'_i;1]^	op U_r[g'_j;1].$$
训练时使用真实父节点计算关系分类损失，联合损失为：
$$L=L_{arc}+L_{rel},\quad
L_{arc}=-
rac1Nsum_ilog p(h_i^{gold}mid i),\quad
L_{rel}=-
rac1Nsum_ilog p(r_i^{gold}mid i,h_i^{gold}).$$
推理时改用解码得到的父节点。只对选定的词对计算关系评分，避免构造较大的 $B	\times T	\times T	\times R$ 张量。

In [ ]:
config = json.loads((ROOT / "configs/full.json").read_text())
中文展示(pd.Series(config["model"], name="完整训练配置").to_frame())
中文展示(Code(inspect.getsource(Biaffine), language="python"))

以下单元对随机初始化的模型执行前向传播、联合损失计算与反向传播。
通过打包变长序列处理不同句长，并将自环和填充父节点的分数设为 $-\infty$。

In [ ]:
untrained = Parser(vocabs, ModelConfig(**config["model"]))
arcs, rel_features = untrained(batch)
loss = untrained.loss(batch, arcs, rel_features)
loss.backward()
print("依存弧张量形状［批次、依存词、父节点］：", tuple(arcs.shape))
print("随机初始化时的损失：", round(loss.item(), 4))
print("模型参数量：", f"{sum(p.numel() for p in untrained.parameters()):,}")
print("词嵌入梯度范数：", round(untrained.word_embedding.weight.grad.norm().item(), 6))
del untrained, arcs, rel_features, loss

基线模型不包含 BiLSTM 或双线性、双仿射参数，而是通过词级 MLP 和加性成对评分预测依存弧：
$$s_{MLP}(i,j)=v^	op	anh(W_d x_i+W_h x_j).$$
其关系分类器同样采用加性 MLP。
三组模型使用相同的单根解码器、数据划分、优化器、最大训练轮数及验证集选择规则。

## 依存树解码
逐词独立选择最高分父节点可能产生环或多个根词。
Chu–Liu/Edmonds 算法先收缩有向环，调整入边分数，求解收缩后的图，再展开恢复。
对于环 $C$，调整后的入边评分为：
$$s'(C,h)=max_{din C}{s(d,h)-s(d,hat h_d)}.$$
普通有向生成树允许多个词直接依附 ROOT。
若松弛最优解存在多个根词，本实现会枚举根词并强制只保留一条 ROOT 边，
再利用可采纳的逐行上界安全剪枝，保持求解结果最优。算法允许交叉弧，即支持非投射依存结构。

In [ ]:
scores = np.array([[0, 0, 0, 0], [4, 0, 9, 1], [4, 9, 0, 1], [8, 1, 1, 0]], dtype=float)
greedy = scores.copy()
np.fill_diagonal(greedy, -np.inf)
print("逐词独立选择的父节点：", [-1] + greedy.argmax(axis=1)[1:].tolist())
heads = decode_mst(scores)
print("单根最大生成树：", heads.tolist())
print("是否为合法依存树：", validate_tree(heads))
print("依存树总得分：", sum(scores[i, heads[i]] for i in range(1, len(heads))))
中文展示(Code(inspect.getsource(decode_mst), language="python"))

独立测试将解码结果与小规模随机图的穷举结果对比，同时覆盖并列最优解和禁用边。
测试还包含单词句、多根候选、有向环、非投射树、填充掩码、推理不读取金标准标签，
以及 CPU 上断点恢复与连续训练结果完全一致的检查。

In [ ]:
import subprocess
test_run = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT,
                          capture_output=True, text=True)
import re
passed = re.search(r"(\d+) passed", test_run.stdout)
print("单元测试通过数量：", passed.group(1) if passed else "请查看测试错误信息")
assert test_run.returncode == 0, test_run.stderr

##  模型训练
完整实验采用 Adam，学习率为 0.001，批大小为 32，梯度范数裁剪阈值为 5，随机种子为 42，
最多训练 30 轮，早停耐心值为 5。使用**包括全部词、比较完整标签的验证集 LAS** 选择最佳模型。
输入、编码器与 MLP 的丢弃率均为 0.33，随机词替换概率为 0.10，并按近似句长组批以减少填充。
最后一轮断点包含优化器和随机数状态，可以在轮次边界恢复训练。
各实验可通过独立进程共享 GPU；记录的训练墙钟时间包含资源竞争。
所有训练结束后，再逐个模型测量推理速度。

下面使用 128 个训练句和 40 个验证句，从零训练一轮。
这只是教学演示，其分数与完整数据集实验的报告分数分开记录。

In [ ]:
demo_config = json.loads((ROOT / "configs/demo.json").read_text())
demo_config.update(device="cpu", epochs=1)
demo_train, demo_dev = splits["train"][:128], splits["dev"][:40]
demo_vocabs = build_vocabs(demo_train)
demo_summary = train("notebook_demo", demo_train, demo_dev, demo_vocabs, demo_config,
                     ModelConfig(**demo_config["model"]), ROOT / "checkpoints/notebook_demo")
中文展示(pd.Series(demo_summary, name="单轮教学训练结果").to_frame())

下面的可选单元可重新运行完整训练、消融对比、最终测试预测、官方评估交叉核验及结果生成。
默认直接读取项目中已经完成的完整实验。查看保存的输出不需要连接外部服务器。

In [ ]:
if RUN_FULL_TRAINING:
    from src.experiment import run
    run(ROOT / "configs/full.json")
assert (RESULTS / "results.json").exists(), "请先运行 python -m src.experiment 生成完整实验结果。"
measured_environment = json.loads((RESULTS / "environment.json").read_text())
中文展示(pd.Series({key: measured_environment[key] for key in ("python", "torch", "cuda", "gpu", "parallel_training")},
                  name="实际完整训练环境").to_frame())
中文展示(Image(filename=str(RESULTS / "figures/training_curves.png")))

## 模型评估

**主要自定义指标**包括标点，并完整比较关系标签，例如 `nsubj:pass` 与 `nsubj` 不相同。
**官方 CoNLL-2018 指标**同样包括标点，但会去除关系子类型。
自定义粗粒度得分会与官方脚本逐项核对；另提供按金标准 UPOS 排除 `PUNCT` 的完整标签得分。
ROOT、多词行和空节点均不进入分母。
由于保留金标准分词，官方 F1 与对应的依存准确率相等。

参考：[官方评估规则](https://universaldependencies.org/conll18/evaluation.html)。
推理速度包括神经评分、CPU 上的 MST 解码和关系预测，不包括磁盘读写及输入编码。

In [ ]:
results = json.loads((RESULTS / "results.json").read_text())
comparison = pd.read_csv(RESULTS / "comparison.csv")
中文展示(comparison.round(2))
中文展示(Image(filename=str(RESULTS / "figures/model_comparison.png")))
metric_rows = []
for result in results:
    for scheme, scores in result["test"].items():
        metric_rows.append({"Model": result["name"], "指标口径": scheme, **scores})
中文展示(pd.DataFrame(metric_rows).round(3))
for result in results:
    for key in ("UAS", "LAS"):
        assert abs(result["test"]["all_coarse"][key] - result["official"][key]) < 1e-8
print("三组模型的自定义粗粒度得分均与官方评估结果一致。")

以下单元针对验证集选出的模型，对保存的完整测试集预测重新执行官方评估。
模型选择在查看测试标签之前已固定；对比表仅描述实验结果，不改变选择规则。

In [ ]:
from src.experiment import official_scores
selection = json.loads((RESULTS / "selection.json").read_text())
selected = selection["selected_model"]
print("已固定的验证集选择：", 中文取值.get(selected, selected))
official = official_scores(ROOT / "data/raw/en_ewt-ud-test.conllu", RESULTS / f"{selected}_test.conllu")
print("重新计算的完整测试集官方得分：", official)

### CoNLL-U 推理
`Predictor.predict(input_conllu_path, output_conllu_path)` 只读取词形及可选的通用词性，
输入中的真实父节点与依存关系可以缺失。
输出保留表层注释、多词条目、词元、词性、形态与其他元信息，清空过期的增强依存字段，
并删除仅用于增强图的空节点。
显式保证 `HEAD=0` 当且仅当 `DEPREL=root`，写出后重新解析并验证基本依存树。

In [ ]:
# 推理前清除金标准父节点和关系，演示无标签输入接口。
from src.data import Sentence
import copy
examples = json.loads((RESULTS / "examples.json").read_text())
demo_sentences = [copy.deepcopy(splits["test"][e["index"]]) for e in examples]
input_path = RESULTS / "demo_unlabeled.conllu"
lines = []
for sentence in demo_sentences:
    for row in sentence.rows:
        if isinstance(row, str):
            lines.append(row)
        elif "." not in row[0]:
            row[6] = row[7] = row[8] = "_"
            lines.append("\t".join(row))
    lines.append("")
input_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
predictor = Predictor(ROOT / "checkpoints/full" / selected / "best.pt", device="cpu")
predictions, speed = predictor.predict(input_path, RESULTS / "demo_predictions.conllu")
assert len(read_conllu(RESULTS / "demo_predictions.conllu")) == 3
print("无标签输入预测成功，句子数为", len(predictions), "。")
print((RESULTS / "demo_predictions.conllu").read_text(encoding="utf-8"))

## 实验对比与错误分析
分析对象是验证集选出的模型。以下分数包括标点，并完整比较关系标签。
依存距离依据金标准父节点计算，即 $|i-h_i|$，其中包括根弧。
短句、中句、长句分别包含 1–10、11–25、26 个及以上句法词。
关系分组采用**金标准关系**的通用部分，但判定预测正确时仍比较完整标签。
各分组均报告参与统计的词数。

In [ ]:
中文展示(Image(filename=str(RESULTS / "figures/error_analysis.png")))
for key in ("distance_group", "length_group", "relation", "oov"):
    print("分组依据：", 中文列名.get(key, key))
    table = pd.read_csv(RESULTS / f"errors_by_{key}.csv")
    中文展示(table.round(2))

下面展示三个完整测试实例，对比金标准树与预测树，并分别标明父节点错误和关系错误。
红色依存弧表示父节点或完整关系标签预测错误。
实例按固定规则选择：测试集中前三个长度为 4–12 且存在错误的句子，不按预设结论挑选。

In [ ]:
from src.report import token_records
error_records = pd.read_csv(RESULTS / "token_errors.csv")
for number, example in enumerate(examples, 1):
    中文展示(Markdown(f"**实例 {number}：** {example['text']}"))
    中文展示(Image(filename=str(RESULTS / f"figures/example_{number}.png")))
    table = error_records[error_records.sentence == example["index"]].copy()
    table["head_error"] = table.gold_head != table.pred_head
    table["relation_error"] = table.gold_rel != table.pred_rel
    中文展示(table[["position", "word", "upos", "gold_head", "pred_head", "gold_rel", "pred_rel",
                   "head_error", "relation_error", "oov"]])
    errors = table[table.head_error | table.relation_error]
    oov_errors = int(errors.oov.sum())
    long_errors = int((errors.distance >= 7).sum())
    中文展示(Markdown(f"本例共有 **{len(errors)}** 个附着或关系错误；"
                     f"其中 **{oov_errors}** 个涉及未登录词，**{long_errors}** 个涉及距离至少为 7 的金标准依存弧。"
                     + ("可进一步检查词汇稀疏是否与这些错误有关。" if oov_errors else
                        "可见词表内的词同样会发生预测错误。")
                     + ("长距离上下文也值得进一步检查。" if long_errors else
                        "这些错误也可能发生在局部附着中。")
                     + "上方金标准与预测结果展示了不同的附着选择；"
                     "仅凭一个实例无法确定错误原因。"))

以下观察直接根据已保存的实测结果生成。
不同分组之间的差异描述本次运行中的现象，不能单独证明依存距离、词性或未登录词导致了某个错误。

In [ ]:
word = next(r for r in results if r["name"] == "word_only")
pos = next(r for r in results if r["name"] == "word_pos")
chosen = next(r for r in results if r["name"] == selected)
delta = pos["test"]["all_full"]["LAS"] - word["test"]["all_full"]["LAS"]
distance = pd.read_csv(RESULTS / "errors_by_distance_group.csv")
length = pd.read_csv(RESULTS / "errors_by_length_group.csv")
relations = pd.read_csv(RESULTS / "errors_by_relation.csv")
worst = relations[relations.Tokens >= 100].nsmallest(4, "LAS")
observations = [
    f"验证集选出的系统为 **{中文取值.get(selected, selected)}**，测试集 UAS 为 "
    f"**{chosen['test']['all_full']['UAS']:.2f}%**，完整标签 LAS 为 **{chosen['test']['all_full']['LAS']:.2f}%**.",
    f"相对仅词形模型，加入金标准词性后测试 LAS 变化 **{delta:+.2f} 个百分点**。",
    f"依存距离为 1 时，LAS 为 **{distance.iloc[0].LAS:.2f}%**, 距离至少为 7 时为 **{distance.iloc[-1].LAS:.2f}%**。",
    f"短句的 LAS 为 **{length.iloc[0].LAS:.2f}%**, 长句的 LAS 为 **{length.iloc[-1].LAS:.2f}%**。",
    "在测试词数不少于 100 的关系类别中，LAS 最低的类别包括："
    + ", ".join(f"`{r.relation}` ({r.LAS:.2f}%, 词数={int(r.Tokens)})" for r in worst.itertuples()) + "."
]
中文展示(Markdown("\n\n".join(observations)))

综上，本项目实现了数据处理、可训练词嵌入、BiLSTM 编码、弧与关系的双仿射评分、
精确单根 MST 解码、带断点的训练、CoNLL-U 推理、官方评估交叉核验，